See skript tekitab kala77 andmebaasi tabeli syntax_morphology_conflicts tabeli.
syntax_morphology_conflicts tabeli sisuks on võimalikud käänete vead morfoloogiliselt märgendamisel.


In [1]:
%load_ext autoreload
%autoreload 2

# syntax_morphology_conflicts andmebaasi tegemine

Skripti töö tulemusena tekitatakse uus andmebaasifail **syntax_morphology_conflicts.db**.
Baasi sisuks on fraasid, mille puhul on alust kahtlustada, et fraasi juure käände märgendamisel on tehtud viga.

Kaetud on **ainult mustribaasis olevad** verbid.

Sisendiks on:

- mallidega katmata transaktsioonide andmebaas: kala77.db


In [2]:
#!pip install tqdm ipywidgets
from datetime import datetime
import os
import sys
import pandas as pd
from pathlib import Path
from estnltk.taggers import VabamorfAnalyzer  # v1.7.3
from smc_helpers import collect_misanalysed_transactions, get_cg_case

# setup kala77.db
ROOT = str(Path(os.getcwd()).parent.parent)
sys.path.append(f"{ROOT}/apriori/v33")

from V33Apriori import V33

In [3]:
# init token analyser
morph_analyzer = VabamorfAnalyzer()


# init V33 
KALA77_DB = ROOT + "/verb_patterns/kala77/kala77.db"
kala77 = V33(file_path=KALA77_DB)

In [4]:
# kõik verbid
verbs = kala77.execute_text("SELECT * FROM verbs").mappings().all()
verbs[0]

{'verb_id': 1, 'verb': 'aasima', 'verb_compound': '', 'pat_ids': '1,2'}

In [5]:
# kõik verbide mallid
patterns = {
    p["pat_id"]: p
    for p in kala77.execute_text("SELECT * FROM patterns").mappings().all()
}
patterns[1]

{'pat_id': 1, 'pattern': 'aasima keda*', 'verb_word': 'aasima', 'verb_compound': '', 'phrase_nr': 1, 'phrase_case': 'part', 'adp': '', 'inf_verb': ''}

In [ ]:
%%time
import json
# tekitame andmebaasi
from sqlalchemy import create_engine
from sqlalchemy.orm import Session
from smc_models import Base, SyntaxMorphologyConflict
from tqdm.notebook import tqdm


# loome andmebaasi
date_time = datetime.now().strftime("%Y%m%d-%H%M%S")
engine = create_engine(f"sqlite:///syntax_morphology_conflicts_{date_time}.db", echo=False)
Base.metadata.create_all(engine)

with Session(engine) as session:
    session.commit()

print("Andmebaas ja tabelid said tehtud.")

# db-sse sisestatakse mitme rea kaupa
batch_size = 10000
batch = []

for v in tqdm(verbs):
    pat_ids = [
        int(id)
        for id in v["pat_ids"].split(",")
        if len(patterns[int(id)]["phrase_case"])
    ]
    if not pat_ids:
        print("Ei leitud fraasimalle analüüsimiseks", v)
        continue

    # verbi kõik transaktsioonid
    kala77.get_transactions(verb=v["verb"], verb_compound=v["verb_compound"])
    
    # klassi sisemuutujas on rohkem informatsiooni
    transactions_raw = kala77._raw_transactions

    # leiame pat_id kaupa, transaktsioone, millel liikmetel oli malliga mittesobiv kääne
    misanalysed_transactions = collect_misanalysed_transactions(pat_ids = pat_ids, patterns=patterns, transactions_raw=transactions_raw)
   
    # verbi malli kaupa
    for pat_id in misanalysed_transactions.keys():
        # mallis nõutud kääne
        pattern_case = patterns[pat_id]["phrase_case"]
        misanalysed_transactions2 = {}
        # kui fraasi liikmete analüüsimisel leiti, et on olemas
        # analüüsi variant, mis sobib malliga käände poolest
        for head_id in misanalysed_transactions[pat_id]:
            for tr in transactions_raw[head_id]:
                possible_cases = [get_cg_case(analysis=a) for a in morph_analyzer.analyze_token(tr['frequent_form'])]
                if pattern_case in possible_cases:
                    tr['possible_cases'] = list(set(possible_cases))
                    misanalysed_transactions2.setdefault(head_id, []).append(tr)

        head_ids = list(misanalysed_transactions2.keys())
        
        # kui ei olnud sobivaid
        if not len(head_ids): continue

        # leiame korraga verbifraaside andemd head_ids massiivi järgi
        phrases = [
            dict(row) for row in kala77.get_phrases_by_head_ids(head_ids=head_ids)
        ]
        
        for i, phrase in enumerate(phrases):
            head_id = phrase['head_id']
            sentence_id = phrase['sentence_id']
            verb_loc = phrase['loc']
            verb_lemma = v['verb']
            verb_compound = v['verb_compound']
            compound_loc = [ row['loc'] for row in transactions_raw[head_id] if  row['deprel'].lower() == 'compound:prt']
            phrase_loc = sorted([verb_loc] + [row['loc'] for row in transactions_raw[head_id]])
            phrase_text = phrase['phrase']
            
            # ühes fraasis võis olla mitu kaheldava analüüsiga sõna
            # iga sõna läheb eraldi reale
            for tr in misanalysed_transactions2[head_id]:
                
                current_morph = tr['feats']
                current_case = tr['case']
                lemma_loc = tr['loc']  # valesti analüüsitud sõna
                lemma_deprel = tr['deprel'].lower() # valesti analüüsitud sõna depreli
                lemma = tr['lemma']
                possible_cases = tr['possible_cases']
                

                conflict_entry = SyntaxMorphologyConflict(
                    pattern_id=pat_id,
                    sentence_id=sentence_id,
                    verb_loc=verb_loc,
                    compound_loc=json.dumps(compound_loc) if len(compound_loc) else None,
            
                    phrase_root_loc= lemma_loc,
                    verb_phrase_loc=json.dumps(phrase_loc),
                    phrase_case=pattern_case,
                    phrase_deprel=lemma_deprel,
                    
                    verb=verb_lemma,
                    verb_compound=verb_compound,
                    phrase=phrase_text,
                    phrase_root_lemma=lemma,
                    current_analysis=current_morph,
                    current_case=current_case,
                    possible_cases=json.dumps(possible_cases)

                )
                
                batch.append(conflict_entry)

                if len(batch) >= batch_size:
                    session.bulk_save_objects(batch)
                    session.commit()
                    batch = []

if batch:
    session.bulk_save_objects(batch)
    session.commit()

session.close()


Andmebaas ja tabelid said tehtud.


  0%|          | 0/1271 [00:00<?, ?it/s]